In [1]:
import sys
sys.path.append("..")            # 저장소 루트 (project 패키지)
sys.path.append("../scripts")    # eval_silver 변환 함수 재사용
from pathlib import Path

In [2]:
BENCH_ID = "AIHub_LowQualityPhoneVoice_phone_8k"
SILVER   = "/data/ASR/BENCHMARK/SILVER/AIHub_LowQualityPhoneVoice/transcript.jsonl"
MODEL    = "openai/whisper-small"
DEVICE   = "cuda:2"              # 실행 직전 nvidia-smi 로 빈 GPU 확인
SAMPLE   = 2000                  # 무작위 샘플 크기 (전체 39,916 중). 저품질이라 변동성 대비 넉넉히
SEED     = 42                    # 재현용

OUT_DIR  = Path(f"../BENCHMARK/results/whisper_small__{BENCH_ID}")

In [3]:
import random
from eval_silver import convert_silver

# 1) 전체 변환 (원본 → GOLD 필드명). 원본 SILVER 는 읽기만 함.
#    빈 정답(o/ 잡음 태그 등) 발화는 셔틀이 자동 제외.
conv_full = OUT_DIR / "_silver_converted" / f"{BENCH_ID}.jsonl"
n = convert_silver(Path(SILVER), conv_full, corpus_id=BENCH_ID)
print(f"전체 변환: {n} samples")

# 2) 무작위 SAMPLE개 추출 (seed 고정 → 재현 가능)
lines = conv_full.read_text(encoding="utf-8").splitlines()
random.seed(SEED)
sample_lines = random.sample(lines, SAMPLE)
conv = conv_full.with_name(f"{BENCH_ID}__sample{SAMPLE}.jsonl")
conv.write_text("\n".join(sample_lines) + "\n", encoding="utf-8")
print(f"무작위 샘플: {len(sample_lines)} samples → {conv}")

  [skip] 정답 전사가 빈 발화 4개 제외
전체 변환: 39912 samples
무작위 샘플: 2000 samples → ../BENCHMARK/results/whisper_small__AIHub_LowQualityPhoneVoice_phone_8k/_silver_converted/AIHub_LowQualityPhoneVoice_phone_8k__sample2000.jsonl


In [4]:
from project.data.adapters.whisper import build_predict_fn

predict_fn = build_predict_fn(
    MODEL, backbone=MODEL,
    language="ko", task="transcribe",
    beam_size=5, batch_size=16, device=DEVICE,
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [5]:
from project.evaluation import evaluate_on_benchmark_suite

results = evaluate_on_benchmark_suite(
    model_name=f"whisper_small__{BENCH_ID}",
    predict_fn=predict_fn,
    benchmark_paths={BENCH_ID: conv},
    out_dir=OUT_DIR,
    batch_size=16,
)
results[BENCH_ID]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take p

CerResult(cer=54.46838040067584, scer=55.58001626306499, wer=84.36223204790956, samples=2000, per_sample_cer=[60.0, 46.15384615384615, 100.0, 50.0, 17.708333333333336, 87.5, 4890.0, 18.0327868852459, 56.25, 18.181818181818183, 27.77777777777778, 15.384615384615385, 27.27272727272727, 11.627906976744185, 31.11111111111111, 19.35483870967742, 27.27272727272727, 21.666666666666668, 16.666666666666664, 26.41509433962264, 17.647058823529413, 25.0, 17.24137931034483, 1650.0, 418.1818181818182, 24.137931034482758, 25.0, 21.21212121212121, 26.126126126126124, 35.294117647058826, 28.767123287671232, 28.30188679245283, 32.83582089552239, 27.173913043478258, 47.82608695652174, 8.695652173913043, 29.72972972972973, 42.857142857142854, 12.0, 28.57142857142857, 42.857142857142854, 33.33333333333333, 3488.2352941176473, 30.0, 92.42424242424242, 41.57303370786517, 37.5, 41.66666666666667, 27.77777777777778, 30.0, 58.620689655172406, 5.555555555555555, 17.391304347826086, 11.11111111111111, 4.761904761

In [6]:
print((OUT_DIR / "evaluation_report.txt").read_text(encoding="utf-8"))

📊 ASR Evaluation Report — whisper_small__AIHub_LowQualityPhoneVoice_phone_8k
   Date: 2026-06-16T17:39:23

## 1. Benchmark Set Results (한국어 CER 표준)
--------------------------------------------------------------------------------
Benchmark                                                  CER (%)   sCER (%)    Samples
--------------------------------------------------------------------------------
AIHub_LowQualityPhoneVoice_phone_8k                          54.47      55.58      2,000
--------------------------------------------------------------------------------
Weighted Average                                             54.47                 2,000

## 2. Slice Analysis (메타 필드별)
--------------------------------------------------------------------------------

### AIHub_LowQualityPhoneVoice_phone_8k
  [by age_group]
  value                   CER (%)    samples
  60대 이상                    98.31         63
  40대                       66.21        365
  50대                       63.97    

In [7]:
import json
import pandas as pd
import jiwer

lines = (OUT_DIR / BENCH_ID / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
df = pd.DataFrame(json.loads(l) for l in lines if l)

df["cer"] = [
    jiwer.cer(r, h) * 100 if r else float("nan")
    for r, h in zip(df["text_normalized"], df["prediction_normalized"])
]

for _, row in df.sort_values("cer", ascending=False).head(20).iterrows():
    print(f"[CER {row.cer:5.1f}] 정답: {row.text_normalized}")
    print(f"             예측: {row.prediction_normalized}\n")

[CER 11066.7] 정답: 따로 안 오시면.
             예측: 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안녕하세요 여러분 안